In [1]:
using LogDensityProblems: LogDensityProblems;
using Distributions
using DelimitedFiles
using Random
using Combinatorics
using MCMCChains

In [2]:
n_schools = 8
y = [28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0] # estimated treatment effects
σ = [15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0]
# Let's define some type that represents the model.
struct RegressionProblem{Ty <: AbstractVector}
	y::Ty
	σ::Ty
end
LogDensityProblems.dimension(model::RegressionProblem) = 10

function LogDensityProblems.logdensity(model::RegressionProblem, parameters::AbstractVector{<:Real})
	μ,τ,θ = parameters[1],parameters[2],parameters[3:end]
	lp = logpdf(Normal(0, 5), μ)
	lp += logpdf(truncated(Cauchy(0, 5),lower=0), τ)

	for i in 1:8
		lp += logpdf(Normal(0, 1), θ[i])
		lp += logpdf(Normal(μ + τ * θ[i], model.σ[i]), model.y[i])
	end
	return lp
end

LogDensityProblems.capabilities(model::RegressionProblem) = LogDensityProblems.LogDensityOrder{0}()
MYmodel = RegressionProblem(y, σ)

RegressionProblem{Vector{Float64}}([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0], [15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])

In [14]:
using AbstractMCMC

struct EnsembleSliceSampler{T<:Float64,A<:Int64} <: AbstractMCMC.AbstractSampler
    "initial length scale"
    μ_init::T
    "number of adapation steps"
    M_adapt::A
    "number of walkers"
    n_walkers::A
    "max number of attempts"
    max_steps::A
end

struct ESState{A<:AbstractMatrix{<:Real},T<:Float64,B<:Int64}
    "current position"
    x::A
    "length scale"
    μ::T
    "iteration"
    t::B
end

struct ESSample{A<:AbstractMatrix{<:Real}}
    "current position"
    x::A # a matrix of dimension n_dim x n_walkers
end


function DifferentialMove(μ::Float64, walker_l::AbstractVector{<:Float64}, walker_m::AbstractVector{<:Float64})
	return μ * (walker_l - walker_m)
end

function tune_lengthscale(t, μ, N_e, N_c, M_adapt)
	N_e = max(1, N_e)

	if t <= M_adapt
		return 2μ * N_e / (N_e + N_c)
	else
		return μ
	end
end

using ju

tune_lengthscale (generic function with 1 method)

In [34]:
Threads.nthreads()

1

In [79]:
function AbstractMCMC.step(
	rng::Random.AbstractRNG,
	model_wrapper::AbstractMCMC.LogDensityModel,
	sampler::EnsembleSliceSampler,
	state::ESState)

	model = model_wrapper.logdensity

	# extract the sampler parameters
	μ = sampler.μ_init
	M_adapt = sampler.M_adapt
	n_walkers = sampler.n_walkers
	max_steps = sampler.max_steps

	# extract current state 
	# walker must ndim* n_walkers
	walkers, μ, t = state.x, state.μ, state.t
	ndim = size(walkers, 1)
	n_wdiv2 = div(n_walkers, 2)

	T = eltype(walkers)
	logdens(x::AbstractVector) = LogDensityProblems.logdensity(model, x)

	# sets 
	next_walkers = Matrix{T}(undef, ndim, n_walkers)
	# set the contraction and expansion counts to zero
	R, L, N_e, N_c = 0.0, 0.0, 0, 0

	# get the log density all the walkers
	logdensities = [logdens(x) for x in eachcol(walkers)]


	# Randomly shuffle the walkers
	walker_indexes = Random.randperm(rng, n_walkers)
	# get the first half of the shuffled walkers
	wl_ind = 1:n_wdiv2

	subset_a = walker_indexes[wl_ind]
	# get the second half of the shuffled walkers
	subset_b = walker_indexes[n_wdiv2+1:end]
	# get the two subsets
	sets = [[subset_a, subset_b], [subset_b, subset_a]]


	Widths = Vector{T}(undef, n_wdiv2)

	logdensities_shrink = Vector{T}(undef, n_wdiv2)
	positions_shrink = Matrix{T}(undef, ndim, n_wdiv2)
	# initialise the log densities for the left and right stepping points
	logdensities_left, logdensities_right = Vector{T}(undef, n_wdiv2), Vector{T}(undef, n_wdiv2)
	position_left, position_right = Matrix{T}(undef, ndim, n_wdiv2), Matrix{T}(undef, ndim, n_wdiv2)

	# iterate over the two sets
	for set in sets
		active, inactive = set

		### DifferentialMove ###
		# get all the permutations of the inactive walkers
		permuts = collect(permutations(inactive, 2))
		# get the number of permutations
		pairs = sample(rng, permuts, n_wdiv2, replace = false)
		# iterate over the pairs
		η = hcat([DifferentialMove(μ, walkers[:, p[1]], walkers[:, p[2]]) for p in pairs]...) # (ndim, n_wdiv2)

		# masks the for the left and right stepping
		mask_left = fill(true, n_wdiv2)
		mask_right = fill(true, n_wdiv2)


		# get the move to the new position
		# draw y position 
		δ = rand(rng, Exponential(1), n_wdiv2)
		Y = logdensities[active] - δ

		# interval for the left stepping point
		L = -rand(rng, n_wdiv2)
		# interval for the right stepping point
		R = L .+ 1
		# initialise the counter
		l = 0

		J = floor.(Int, max_steps .* rand(rng, n_wdiv2))
		K = max_steps - 1 .- J

		# stepping out procedure
		while size(mask_left[mask_left], 1) > 0 || size(mask_right[mask_right], 1) > 0

			if size(mask_left[mask_left], 1) > 0
				l += 1
			end
			if size(mask_right[mask_right], 1) > 0
				l += 1
			end
			if l > max_steps
				error("Max steps reached in stepping out")
			end

			for j in wl_ind[mask_left]
				if J[j] < 1
					mask_left[j] = false
				end
			end
			for j in wl_ind[mask_right]
				if K[j] < 1
					mask_right[j] = false
				end
			end
			position_left[:, mask_left] = L[mask_left]' .* η[:, mask_left] + walkers[:, active][:, mask_left]
			position_right[:, mask_right] = R[mask_right]' .* η[:, mask_right] + walkers[:, active][:, mask_right]

			if size(position_left[:, mask_left], 1) + size(position_right[:, mask_right], 1) < 0
				logdensities_left[mask_left] = []
				logdensities_right[mask_right] = []
				l -= 1
			else
				nl = size(position_left[:, mask_left], 2)
				nr = size(position_right[:, mask_right], 2)
				logdensities_left[mask_left] = [logdens(position_left[:, mask_left][:, i]) for i in 1:nl]
				logdensities_right[mask_right] = [logdens(position_right[:, mask_right][:, i]) for i in 1:nr]
			end
			for j in wl_ind[mask_left]
				if Y[j] < logdensities_left[j]
					L[j] -= 1
					N_e += 1
					J[j] -= 1
				else
					mask_left[j] = false
				end
			end
			for j in wl_ind[mask_right]
				if Y[j] < logdensities_right[j]
					R[j] += 1
					N_e += 1
					K[j] -= 1
				else
					mask_right[j] = false
				end
			end
		end

		## shrink the interval##
		mask = fill(true, n_wdiv2)
		l = 0
		while size(mask[mask], 1) > 0

			Widths[mask] = rand(rng, Uniform(), size(mask[mask])) .* (R[mask] - L[mask]) .+ L[mask]


			positions_shrink[:, mask] = Widths[mask]' .* η[:, mask] + walkers[:, active][:, mask]
			logdensities_shrink[mask] = [logdens(positions_shrink[:, mask][:, i]) for i in 1:size(positions_shrink[:, mask], 2)]

			for j in wl_ind[mask]
				if Y[j] < logdensities_shrink[j]
					mask[j] = false
				else
					if Widths[j] < 0.0
						L[j] = Widths[j]
						N_c += 1
					else
						R[j] = Widths[j]
						N_c += 1
					end
				end
			end
			l += 1
			if l > max_steps
				error("Max steps reached in shrink")
			end
		end

		# update the walker
		walkers[:, active] = positions_shrink
		logdensities[active] = logdensities_shrink

		next_walkers[:, active] = positions_shrink
	end


	μ = tune_lengthscale(t, μ, N_e, N_c, M_adapt)
	t += 1
	state_new = ESState(next_walkers, μ, t)
	return ESSample(next_walkers), state_new
end


In [80]:
rng = Random.default_rng(89)
ndims = 10
n_walkers = 80#2 * ndims
sampler = EnsembleSliceSampler(1.0, 50, n_walkers, 10_000)

# initialize the walkers
m = AbstractMCMC.LogDensityModel(MYmodel).logdensity
init_start = randn(rng, ndims,n_walkers)
init_start[2,:] = abs.(init_start[2,:])
logdens = [ LogDensityProblems.logdensity(m,init_start[:,i]) for i in 1:n_walkers]
@assert all(logdens .!= -Inf) "Initial positions are not valid"

state = ESState(init_start, 1.0, 1)

ESState{Matrix{Float64}, Float64, Int64}([0.9757093195008314 2.1187179038448303 … 2.1322497449158826 -0.48144496586526797; 1.7833159557404525 1.8127181247075574 … 1.8685450762702924 0.10567819572844912; … ; 1.3636522482738922 0.6900128227057866 … 0.02403393279266425 1.8039249327112716; -0.9345883723993237 1.1457483361382028 … -1.1175439387518067 1.5396479216284036], 1.0, 1)

In [81]:
rng = Random.default_rng(2)

x_next, state_next = AbstractMCMC.step(
    rng,
    AbstractMCMC.LogDensityModel(MYmodel),
    sampler,
    state
)

(ESSample{Matrix{Float64}}([1.0514101697342597 2.052229845878772 … 2.9425896226514148 -0.4448363868991078; 1.6899243315668313 1.8470510176338317 … 2.0628200903673917 0.10446671997901079; … ; 1.530979979919427 0.4953274983475181 … 0.30173978363181414 1.7246341190268883; -0.8486542121872811 0.9727523115429005 … -0.8348388800777343 1.5898325537068814]), ESState{Matrix{Float64}, Float64, Int64}([1.0514101697342597 2.052229845878772 … 2.9425896226514148 -0.4448363868991078; 1.6899243315668313 1.8470510176338317 … 2.0628200903673917 0.10446671997901079; … ; 1.530979979919427 0.4953274983475181 … 0.30173978363181414 1.7246341190268883; -0.8486542121872811 0.9727523115429005 … -0.8348388800777343 1.5898325537068814], 0.8531468531468531, 2))

In [82]:
samples = sample(MYmodel, sampler, 20_000; initial_state=state, progress=true)

Sampling   0%|                                          |  ETA: N/A
Sampling   0%|▎                                         |  ETA: 0:00:27
Sampling   1%|▍                                         |  ETA: 0:00:29
Sampling   2%|▋                                         |  ETA: 0:00:30
Sampling   2%|▉                                         |  ETA: 0:00:29
Sampling   2%|█                                         |  ETA: 0:00:29
Sampling   3%|█▎                                        |  ETA: 0:00:30
Sampling   4%|█▌                                        |  ETA: 0:00:29
Sampling   4%|█▋                                        |  ETA: 0:00:29
Sampling   4%|█▉                                        |  ETA: 0:00:29
Sampling   5%|██▏                                       |  ETA: 0:00:28
Sampling   6%|██▎                                       |  ETA: 0:00:28
Sampling   6%|██▌                                       |  ETA: 0:00:28
Sampling   6%|██▊                                       |  ETA: 0:00

20000-element Vector{ESSample{Matrix{Float64}}}:
 ESSample{Matrix{Float64}}([1.9681426038716108 1.9640805699483188 … 3.5483449531114974 -1.1151306394473892; 1.723922444788192 1.7527545918537712 … 2.048904545019697 0.42366218313715326; … ; 1.3490765773605775 0.18614493723157557 … 0.08029092183523706 2.2206145946623654; -0.7791440499497997 0.40895552826629833 … -0.6396638301179951 0.7447349335547848])
 ESSample{Matrix{Float64}}([1.909372385460709 4.605062394037143 … 4.451732592810464 -0.6954395359707692; 1.7402099453903406 1.84135537522393 … 1.959103792908685 0.5552833479576326; … ; 1.5077997173798716 0.220454017082277 … 0.8892509192077238 1.830015427719124; -0.8749366862046798 -0.10425623994796407 … -0.40485670234926985 0.9606446329801488])
 ESSample{Matrix{Float64}}([1.648131895321588 5.738602294253321 … 4.220361454258072 -0.7318757669941435; 2.014709037120223 1.9717198304967236 … 1.9812810314046765 0.5584076194299231; … ; 1.177860642982 0.6238830145745468 … 0.8042062738958305 1.823437

In [43]:
samples = sample(MYmodel, sampler, 20_000; initial_state=state, progress=true)

Sampling   0%|                                          |  ETA: N/A
Sampling   0%|▎                                         |  ETA: 0:00:38
Sampling   1%|▍                                         |  ETA: 0:00:32
Sampling   2%|▋                                         |  ETA: 0:00:31
Sampling   2%|▉                                         |  ETA: 0:00:31
Sampling   2%|█                                         |  ETA: 0:00:29
Sampling   3%|█▎                                        |  ETA: 0:00:29
Sampling   4%|█▌                                        |  ETA: 0:00:29
Sampling   4%|█▋                                        |  ETA: 0:00:28
Sampling   4%|█▉                                        |  ETA: 0:00:28
Sampling   5%|██▏                                       |  ETA: 0:00:28
Sampling   6%|██▎                                       |  ETA: 0:00:27
Sampling   6%|██▌                                       |  ETA: 0:00:27
Sampling   6%|██▊                                       |  ETA: 0:00

20000-element Vector{ESSample{Matrix{Float64}}}:
 ESSample{Matrix{Float64}}([1.9304219510870946 0.4882497894018421 … -0.283267219600136 -0.8653911241225911; 2.1006995261463954 0.0004896463177042343 … 0.9771228332668338 1.4534540065250667; … ; 0.9126231974195053 -0.9377151450286988 … -0.3378199429612241 1.4309586534679963; 0.2181605282606602 -1.8240938952742012 … -0.5549384405552049 1.6337613573012169])
 ESSample{Matrix{Float64}}([1.836965948636774 0.37943549479418814 … -0.2847255701306471 -0.13090752378582038; 2.1032681298590004 0.11260181489041472 … 0.9759373270915797 0.5953159218488013; … ; 0.9889987311886711 -0.7881564012697138 … -0.3409672380266789 1.063975322424384; 0.16947767859902502 -1.522700190193023 … -0.553433473958905 1.2992377822762586])
 ESSample{Matrix{Float64}}([2.14459558965469 0.7613045399133387 … -0.2624567679798313 -0.46776865632103554; 2.234534925493082 0.34825233301136665 … 0.9597179039930801 0.6392532084178908; … ; 0.8083188433353348 -0.9386855665414371 … 0.12974

In [32]:
samples_matrix = stack(sample -> sample.x, samples);
samples_matrix = permutedims(samples_matrix, [3, 1,2])

20000×10×50 Array{Float64, 3}:
[:, :, 1] =
 -0.0405872  0.756446   1.24473   …  -0.378845  2.4268    -0.761012
 -0.679756   0.527543   1.40625      -0.571109  2.71421   -0.553725
 -0.863015   0.547475   1.08401      -0.48327   3.08955   -0.705493
 -1.0446     0.523175   1.00696      -0.468992  3.03889   -0.545272
 -0.803981   0.0358689  0.32305      -0.943177  2.11624   -0.691515
 -0.950743   0.0819707  0.357369  …  -0.961956  2.17228   -0.710533
 -0.420551   0.228693   0.690273     -0.263457  1.77247   -0.333123
 -2.31811    0.0607568  0.352801     -0.560545  1.54187   -1.26147
 -2.20443    0.0991428  0.335738     -0.580745  1.48764   -1.23865
 -1.64232    0.012812   0.423881     -0.784161  1.13753   -0.925138
  ⋮                               ⋱                       
  6.2478     1.63353    1.43509       1.94954   0.451224  -0.222623
  5.14271    1.23328    1.17376       1.9837    0.845254  -0.374767
  5.14993    1.44791    1.02245       1.96085   0.961875  -0.300306
  6.00016    1.1

In [33]:
chn = Chains(samples_matrix, ["μ", "τ", "θ[1]", "θ[2]", "θ[3]", "θ[4]", "θ[5]", "θ[6]", "θ[7]", "θ[8]"])

Chains MCMC chain (20000×10×50 Array{Float64, 3}):

Iterations        = 1:1:20000
Number of chains  = 50
Samples per chain = 20000
parameters        = μ, τ, θ[1], θ[2], θ[3], θ[4], θ[5], θ[6], θ[7], θ[8]

Summary Statistics
  parameters      mean       std      mcse     ess_bulk     ess_tail      rhat ⋯
      Symbol   Float64   Float64   Float64      Float64      Float64   Float64 ⋯

           μ    4.3879    3.3103    0.0165   40398.1183   71395.9926    1.0011 ⋯
           τ    3.5934    3.1794    0.0213   34205.5190   25466.9839    1.0016 ⋯
        θ[1]    0.3182    0.9892    0.0049   41582.5999   77501.8925    1.0013 ⋯
        θ[2]    0.0976    0.9370    0.0046   42237.1328   76891.0066    1.0015 ⋯
        θ[3]   -0.0920    0.9658    0.0047   41348.6763   79233.9641    1.0007 ⋯
        θ[4]    0.0557    0.9391    0.0045   43514.3849   79509.7865    1.0010 ⋯
        θ[5]   -0.1570    0.9267    0.0046   41290.6138   71466.3503    1.0013 ⋯
        θ[6]   -0.0624    0.9448    0.0046   4

# Now with turing.jl

In [10]:
function AbstractMCMC.step(
    rng::Random.AbstractRNG,
    model_wrapper::AbstractMCMC.LogDensityModel,
    ::EnsembleSliceSampler;
    kwargs...)
    println("step")
    model = model_wrapper.logdensity
    nwalkers = sampler.n_walkers
    ndims = LogDensityProblems.dimension(model)
    x = randn(rng,nwalkers,ndims)
    x[:,2] = abs.(x[:,2])
    return ESSample(x), ESState(x,1.0,1)
end

In [11]:
using Turing
@model function school_reparam(y::AbstractVector{<:Float64}, σ::AbstractVector{<:Float64}, n_schools::Int64=8)
    μ ~ Normal(0, 5)
    τ ~ truncated(Cauchy(0, 5), lower = 0)
    θ ~ filldist(Normal(0, 1), n_schools)
    for i in 1:n_schools
        y[i] ~ Normal(μ + τ * θ[i], σ[i])
    end
end

mymod = school_reparam(y, σ)
Turing.Inference.getparams(::Turing.Model, sample::ESSample) = sample.x

In [12]:
chain = sample(mymod, externalsampler(sampler), 10)

Sampling   0%|                                          |  ETA: N/A


step


Sampling 100%|██████████████████████████████████████████| Time: 0:00:06


MethodError: MethodError: no method matching unflatten(::DynamicPPL.TypedVarInfo{@NamedTuple{μ::DynamicPPL.Metadata{Dict{AbstractPPL.VarName{:μ, typeof(identity)}, Int64}, Vector{Normal{Float64}}, Vector{AbstractPPL.VarName{:μ, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}, τ::DynamicPPL.Metadata{Dict{AbstractPPL.VarName{:τ, typeof(identity)}, Int64}, Vector{Truncated{Cauchy{Float64}, Continuous, Float64, Float64, Nothing}}, Vector{AbstractPPL.VarName{:τ, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}, θ::DynamicPPL.Metadata{Dict{AbstractPPL.VarName{:θ, typeof(identity)}, Int64}, Vector{DistributionsAD.TuringScalMvNormal{Vector{Float64}, Float64}}, Vector{AbstractPPL.VarName{:θ, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}}, Float64}, ::Matrix{Float64})

Closest candidates are:
  unflatten(::DynamicPPL.TypedVarInfo, !Matched::NamedTuple)
   @ Turing ~/.julia/packages/Turing/QN7BL/src/mcmc/Inference.jl:165
  unfl